In [8]:
# import json

# INPUT_FILE = "data/02_guided_assigned.json"
# OUTPUT_FILE = "data/guided_assigned_clean.json"

# KEEP_COLUMNS = [
#     "chunk_id",
#     "doc_id",
#     "domain",
#     "content",
#     "question_type",
#     "question",
#     "answer",
# ]

# with open(INPUT_FILE, "r", encoding="utf-8") as f:
#     data = json.load(f)

# clean_data = [
#     {key: item.get(key) for key in KEEP_COLUMNS}
#     for item in data
# ]

# with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
#     json.dump(clean_data, f, ensure_ascii=False, indent=2)

# print(f"Input : {len(data):,}")
# print(f"Output: {len(clean_data):,}")
# print(f"Saved : {OUTPUT_FILE}")

# kiểu hallu

## config 

In [9]:
import json
import random
import re

from pathlib import Path
from collections import Counter

INPUT_FILE = Path("data/guided_assigned_clean.json")
OUTPUT_FILE = Path("data/guided_assigned_label.json")

SEED = 42
random.seed(SEED)

LABEL_INTRINSIC = "intrinsic_hallucination"
LABEL_EXTRINSIC = "extrinsic_hallucination"
LABEL_MIXED = "mixed_hallucination"

KEEP_FIELDS = [
    "chunk_id",
    "doc_id",
    "domain",
    "content",
    "question_type",
    "question",
    "answer",
]

In [10]:
UNANSWERABLE_PATTERNS = [
    # Context / ngữ cảnh không có thông tin
    r"\bcontext\s+không\s+cung\s+cấp\b",
    r"\bcontext\s+không\s+có\b",
    r"\bcontext\s+không\s+đề\s+cập\b",
    r"\bcontext\s+không\s+nêu\b",
    r"\bcontext\s+không\s+cho\s+biết\b",

    r"\bngữ\s+cảnh\s+không\s+cung\s+cấp\b",
    r"\bngữ\s+cảnh\s+không\s+có\b",
    r"\bngữ\s+cảnh\s+không\s+đề\s+cập\b",
    r"\bngữ\s+cảnh\s+không\s+nêu\b",
    r"\bngữ\s+cảnh\s+không\s+cho\s+biết\b",

    # Thiếu thông tin
    r"\bkhông\s+cung\s+cấp\s+đủ\s+thông\s+tin\b",
    r"\bkhông\s+cung\s+cấp\s+thông\s+tin\b",
    r"\bkhông\s+cung\s+cấp\b.*\bcụ\s+thể\b",
    r"\bkhông\s+có\s+đủ\s+thông\s+tin\b",
    r"\bkhông\s+đủ\s+thông\s+tin\b",
    r"\bkhông\s+có\s+thông\s+tin\b",
    r"\bthiếu\s+thông\s+tin\b",
    r"\bkhông\s+có\s+dữ\s+liệu\b",

    # Không thể xác định
    r"\bkhông\s+thể\s+trả\s+lời\b",
    r"\bkhông\s+thể\s+xác\s+định\b",
    r"\bkhông\s+xác\s+định\s+được\b",
    r"\bkhông\s+thể\s+kết\s+luận\b",
    r"\bkhông\s+thể\s+suy\s+ra\b",

    # Không được đề cập
    r"\bkhông\s+được\s+đề\s+cập\b",
    r"\bkhông\s+được\s+nêu\b",
    r"\bkhông\s+được\s+cung\s+cấp\b",
    r"\bkhông\s+đề\s+cập\b",
    r"\bkhông\s+nêu\b",
    r"\bkhông\s+cho\s+biết\b",
]


def normalize_text(text):
    if text is None:
        return ""

    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def is_unanswerable(answer):
    text = normalize_text(answer)

    if not text:
        return True

    return any(
        re.search(pattern, text)
        for pattern in UNANSWERABLE_PATTERNS
    )

In [11]:
tests = [
    "Context không cung cấp địa chỉ cụ thể để đăng nhập hệ thống.",
    "Context không đề cập đến doanh thu năm 2024.",
    "Không thể xác định thời gian cụ thể từ thông tin được cung cấp.",
    "Trách nhiệm.",
]

for text in tests:
    print(is_unanswerable(text), "->", text)

True -> Context không cung cấp địa chỉ cụ thể để đăng nhập hệ thống.
True -> Context không đề cập đến doanh thu năm 2024.
True -> Không thể xác định thời gian cụ thể từ thông tin được cung cấp.
False -> Trách nhiệm.


## split sentence

In [12]:
def split_sentences(text):
    if text is None:
        return []

    text = str(text).strip()

    if not text:
        return []

    text = re.sub(r"\s+", " ", text)

    parts = re.split(
        r"(?<=[.!?;])\s+",
        text
    )

    return [
        part.strip(" \t\r\n.;")
        for part in parts
        if part.strip(" \t\r\n.;")
    ]


def looks_like_independent_clause(text):
    text = text.strip()

    if not text:
        return False

    tokens = text.split()

    if len(tokens) < 3:
        return False

    predicate_patterns = [
        r"\blà\b",
        r"\bcó\b",
        r"\bđạt\b",
        r"\bđược\b",
        r"\bthuộc\b",
        r"\bgồm\b",
        r"\bbao\s+gồm\b",
        r"\bgiúp\b",
        r"\bcho\s+phép\b",
        r"\byêu\s+cầu\b",
        r"\bquy\s+định\b",
        r"\bthực\s+hiện\b",
        r"\bdiễn\s+ra\b",
        r"\bbắt\s+đầu\b",
        r"\bkết\s+thúc\b",
        r"\btăng\b",
        r"\bgiảm\b",
        r"\bchiếm\b",
        r"\bcó\s+thể\b",
        r"\bphải\b",
    ]

    lowered = normalize_text(text)

    return (
        len(tokens) >= 6
        or any(
            re.search(pattern, lowered)
            for pattern in predicate_patterns
        )
    )

In [13]:
def split_claims_simple(answer):
    if answer is None:
        return []

    text = str(answer).strip()

    if not text:
        return []

    sentences = split_sentences(text)

    claims = []

    for sentence in sentences:

        candidate_parts = re.split(
            r"\s+(?:và|đồng thời|trong khi|còn|ngoài ra)\s+",
            sentence,
            flags=re.IGNORECASE
        )

        if len(candidate_parts) >= 2:

            candidate_parts = [
                part.strip(" ,.;")
                for part in candidate_parts
                if part.strip(" ,.;")
            ]

            independent_parts = [
                part
                for part in candidate_parts
                if looks_like_independent_clause(part)
            ]

            if len(independent_parts) >= 2:
                claims.extend(candidate_parts)
                continue

        claims.append(sentence)

    return claims

## priority target

In [14]:
def build_priority(primary_label):
    all_labels = [
        LABEL_INTRINSIC,
        LABEL_EXTRINSIC,
        LABEL_MIXED,
    ]

    remaining = [
        label
        for label in all_labels
        if label != primary_label
    ]

    random.shuffle(remaining)

    return [
        primary_label,
        *remaining
    ]


def assign_target_hallucination_label_priority(answer):

    # ========================================================
    # 1. Unanswerable -> extrinsic
    # ========================================================

    if is_unanswerable(answer):

        return build_priority(
            LABEL_EXTRINSIC
        )

    # ========================================================
    # 2. Count claims
    # ========================================================

    claims = split_claims_simple(answer)
    claim_count = len(claims)

    # >= 2 claims
    if claim_count >= 2:

        primary = random.choice([
            LABEL_INTRINSIC,
            LABEL_EXTRINSIC,
            LABEL_MIXED,
        ])

    # 1 claim
    else:

        primary = random.choice([
            LABEL_INTRINSIC,
            LABEL_EXTRINSIC,
        ])

    return build_priority(primary)

In [15]:
with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:
    data = json.load(f)

if not isinstance(data, list):
    raise ValueError(
        "Input JSON phải là list các QA samples."
    )

print(f"Loaded: {len(data):,} samples")

output = []

for item in data:

    new_item = {
        key: item.get(key)
        for key in KEEP_FIELDS
    }

    answer = item.get("answer", "")

    new_item[
        "target_hallucination_label_priority"
    ] = assign_target_hallucination_label_priority(
        answer
    )

    output.append(new_item)

print(f"Processed: {len(output):,} samples")



Loaded: 14,883 samples
Processed: 14,883 samples


In [18]:
OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"Saved -> {OUTPUT_FILE.resolve()}")

Saved -> /home/trang/study/khoaluan/KLTN/src/create_hallu/qa/qa_hallu_full/data/guided_assigned_label.json


# ver 2

In [22]:
# -*- coding: utf-8 -*-

import json
import random
import re

from pathlib import Path
from collections import Counter, defaultdict


# ============================================================
# 1. CONFIG
# ============================================================

INPUT_FILE = Path("data/guided_assigned_clean.json")
OUTPUT_FILE = Path("data/guided_assigned_label.json")

SEED = 42
random.seed(SEED)

# Số lượng mixed mong muốn
TARGET_MIXED = 2500

LABEL_INTRINSIC = "intrinsic_hallucination"
LABEL_EXTRINSIC = "extrinsic_hallucination"
LABEL_MIXED = "mixed_hallucination"

ALL_LABELS = [
    LABEL_INTRINSIC,
    LABEL_EXTRINSIC,
    LABEL_MIXED,
]

KEEP_FIELDS = [
    "chunk_id",
    "doc_id",
    "domain",
    "content",
    "question_type",
    "question",
    "answer",
]


# ============================================================
# 2. UNANSWERABLE PATTERNS
# ============================================================

UNANSWERABLE_PATTERNS = [
    # Context / ngữ cảnh không có thông tin
    r"\bcontext\s+không\s+cung\s+cấp\b",
    r"\bcontext\s+không\s+có\b",
    r"\bcontext\s+không\s+đề\s+cập\b",
    r"\bcontext\s+không\s+nêu\b",
    r"\bcontext\s+không\s+cho\s+biết\b",

    r"\bngữ\s+cảnh\s+không\s+cung\s+cấp\b",
    r"\bngữ\s+cảnh\s+không\s+có\b",
    r"\bngữ\s+cảnh\s+không\s+đề\s+cập\b",
    r"\bngữ\s+cảnh\s+không\s+nêu\b",
    r"\bngữ\s+cảnh\s+không\s+cho\s+biết\b",

    # Thiếu thông tin
    r"\bkhông\s+cung\s+cấp\s+đủ\s+thông\s+tin\b",
    r"\bkhông\s+cung\s+cấp\s+thông\s+tin\b",
    r"\bkhông\s+cung\s+cấp\b.*\bcụ\s+thể\b",
    r"\bkhông\s+có\s+đủ\s+thông\s+tin\b",
    r"\bkhông\s+đủ\s+thông\s+tin\b",
    r"\bkhông\s+có\s+thông\s+tin\b",
    r"\bthiếu\s+thông\s+tin\b",
    r"\bkhông\s+có\s+dữ\s+liệu\b",

    # Không thể xác định
    r"\bkhông\s+thể\s+trả\s+lời\b",
    r"\bkhông\s+thể\s+xác\s+định\b",
    r"\bkhông\s+xác\s+định\s+được\b",
    r"\bkhông\s+thể\s+kết\s+luận\b",
    r"\bkhông\s+thể\s+suy\s+ra\b",

    # Không được đề cập
    r"\bkhông\s+được\s+đề\s+cập\b",
    r"\bkhông\s+được\s+nêu\b",
    r"\bkhông\s+được\s+cung\s+cấp\b",
    r"\bkhông\s+đề\s+cập\b",
    r"\bkhông\s+nêu\b",
    r"\bkhông\s+cho\s+biết\b",

    # Không có cơ sở
    r"\bkhông\s+có\s+cơ\s+sở\b.*\b(?:xác\s+định|trả\s+lời|kết\s+luận)\b",
]


# ============================================================
# 3. NORMALIZE TEXT
# ============================================================

def normalize_text(text):
    if text is None:
        return ""

    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# ============================================================
# 4. DETECT UNANSWERABLE
# ============================================================

def is_unanswerable(answer):
    text = normalize_text(answer)

    if not text:
        return True

    return any(
        re.search(pattern, text, flags=re.IGNORECASE)
        for pattern in UNANSWERABLE_PATTERNS
    )


# ============================================================
# 5. SENTENCE SPLITTING
# ============================================================

def split_sentences(text):
    if text is None:
        return []

    text = str(text).strip()

    if not text:
        return []

    text = re.sub(r"\s+", " ", text)

    parts = re.split(
        r"(?<=[.!?;])\s+",
        text
    )

    return [
        part.strip(" \t\r\n.;")
        for part in parts
        if part.strip(" \t\r\n.;")
    ]


# ============================================================
# 6. CHECK INDEPENDENT CLAUSE
# ============================================================

def looks_like_independent_clause(text):
    text = text.strip()

    if not text:
        return False

    tokens = text.split()

    if len(tokens) < 3:
        return False

    predicate_patterns = [
        r"\blà\b",
        r"\bcó\b",
        r"\bđạt\b",
        r"\bđược\b",
        r"\bthuộc\b",
        r"\bgồm\b",
        r"\bbao\s+gồm\b",
        r"\bgiúp\b",
        r"\bcho\s+phép\b",
        r"\byêu\s+cầu\b",
        r"\bquy\s+định\b",
        r"\bthực\s+hiện\b",
        r"\bdiễn\s+ra\b",
        r"\bbắt\s+đầu\b",
        r"\bkết\s+thúc\b",
        r"\btăng\b",
        r"\bgiảm\b",
        r"\bchiếm\b",
        r"\bcó\s+thể\b",
        r"\bphải\b",
    ]

    lowered = normalize_text(text)

    # Câu tương đối dài thì coi như có khả năng là clause độc lập
    if len(tokens) >= 6:
        return True

    return any(
        re.search(pattern, lowered)
        for pattern in predicate_patterns
    )


# ============================================================
# 7. SIMPLE CLAIM SPLITTING
# ============================================================

def split_claims_simple(answer):
    if answer is None:
        return []

    text = str(answer).strip()

    if not text:
        return []

    sentences = split_sentences(text)

    claims = []

    for sentence in sentences:

        candidate_parts = re.split(
            r"\s+(?:và|đồng thời|trong khi|còn|ngoài ra)\s+",
            sentence,
            flags=re.IGNORECASE
        )

        if len(candidate_parts) >= 2:

            candidate_parts = [
                part.strip(" ,.;")
                for part in candidate_parts
                if part.strip(" ,.;")
            ]

            independent_parts = [
                part
                for part in candidate_parts
                if looks_like_independent_clause(part)
            ]

            # Chỉ split nếu có ít nhất 2 clause độc lập
            if len(independent_parts) >= 2:
                claims.extend(candidate_parts)
                continue

        claims.append(sentence)

    return claims


# ============================================================
# 8. BUILD PRIORITY
# ============================================================

def build_priority(primary_label):
    """
    Primary đứng đầu.
    Hai label còn lại được random thứ tự làm fallback.
    """

    remaining = [
        label
        for label in ALL_LABELS
        if label != primary_label
    ]

    random.shuffle(remaining)

    return [
        primary_label,
        *remaining,
    ]


# ============================================================
# 9. LARGEST REMAINDER ALLOCATION
# ============================================================

def allocate_proportionally(
    total_target,
    group_sizes,
    capacities=None
):
    """
    Chia total_target theo tỷ lệ kích thước từng group.

    Ví dụ:
        education 30%
        finance   20%
        legal     25%
        ...

    thì TARGET_MIXED cũng được chia gần theo tỷ lệ đó.

    capacities:
        giới hạn tối đa có thể assign trong từng group.

    Trả về:
        dict {
            domain: assigned_count
        }
    """

    groups = list(group_sizes.keys())

    total_size = sum(
        group_sizes.values()
    )

    if total_size == 0:
        return {
            group: 0
            for group in groups
        }

    if capacities is None:
        capacities = {
            group: float("inf")
            for group in groups
        }

    # --------------------------------------------------------
    # Raw proportional quota
    # --------------------------------------------------------

    raw = {
        group: (
            total_target
            * group_sizes[group]
            / total_size
        )
        for group in groups
    }

    # --------------------------------------------------------
    # Floor first
    # --------------------------------------------------------

    allocation = {}

    for group in groups:

        base = int(raw[group])

        allocation[group] = min(
            base,
            capacities.get(
                group,
                float("inf")
            )
        )

    assigned = sum(
        allocation.values()
    )

    remaining = (
        total_target - assigned
    )

    # --------------------------------------------------------
    # Largest remainder
    # --------------------------------------------------------

    remainder_order = sorted(
        groups,
        key=lambda group: (
            raw[group] - int(raw[group])
        ),
        reverse=True
    )

    # --------------------------------------------------------
    # Redistribute until target reached
    # --------------------------------------------------------

    while remaining > 0:

        changed = False

        for group in remainder_order:

            capacity = capacities.get(
                group,
                float("inf")
            )

            if allocation[group] >= capacity:
                continue

            allocation[group] += 1
            remaining -= 1
            changed = True

            if remaining == 0:
                break

        if not changed:
            break

    return allocation


# ============================================================
# 10. LOAD DATA
# ============================================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:
    data = json.load(f)

if not isinstance(data, list):
    raise ValueError(
        "Input JSON phải là list các QA samples."
    )

print("=" * 80)
print(f"Loaded: {len(data):,} samples")
print("=" * 80)


# ============================================================
# 11. BASIC DOMAIN INFORMATION
# ============================================================

domain_indices = defaultdict(list)

for idx, item in enumerate(data):

    domain = (
        item.get("domain")
        or "unknown"
    )

    domain_indices[domain].append(idx)


domain_sizes = {
    domain: len(indices)
    for domain, indices
    in domain_indices.items()
}


print("\nDOMAIN DISTRIBUTION")
print("-" * 80)

for domain in sorted(domain_sizes):

    count = domain_sizes[domain]
    pct = count / len(data) * 100

    print(
        f"{domain:20s}: "
        f"{count:6,d} "
        f"({pct:6.2f}%)"
    )


# ============================================================
# 12. CLASSIFY SAMPLES
# ============================================================

unanswerable_indices = []
single_claim_indices = []
multi_claim_indices = []

# Per-domain
unanswerable_by_domain = defaultdict(list)
single_by_domain = defaultdict(list)
multi_by_domain = defaultdict(list)

claim_count_map = {}

for idx, item in enumerate(data):

    answer = item.get(
        "answer",
        ""
    )

    domain = (
        item.get("domain")
        or "unknown"
    )

    # --------------------------------------------------------
    # Unanswerable
    # --------------------------------------------------------

    if is_unanswerable(answer):

        unanswerable_indices.append(idx)
        unanswerable_by_domain[
            domain
        ].append(idx)

        claim_count_map[idx] = 0

        continue

    # --------------------------------------------------------
    # Count claims
    # --------------------------------------------------------

    claims = split_claims_simple(
        answer
    )

    claim_count = len(claims)

    claim_count_map[idx] = (
        claim_count
    )

    # --------------------------------------------------------
    # Multi claim
    # --------------------------------------------------------

    if claim_count >= 2:

        multi_claim_indices.append(
            idx
        )

        multi_by_domain[
            domain
        ].append(idx)

    # --------------------------------------------------------
    # Single claim
    # --------------------------------------------------------

    else:

        single_claim_indices.append(
            idx
        )

        single_by_domain[
            domain
        ].append(idx)


print()
print("=" * 80)
print("SAMPLE TYPE DISTRIBUTION")
print("=" * 80)

print(
    f"Unanswerable : "
    f"{len(unanswerable_indices):,}"
)

print(
    f"1 claim      : "
    f"{len(single_claim_indices):,}"
)

print(
    f">= 2 claims  : "
    f"{len(multi_claim_indices):,}"
)


# ============================================================
# 13. DOMAIN DETAIL BEFORE ASSIGNMENT
# ============================================================

print()
print("=" * 80)
print("DOMAIN / CLAIM DISTRIBUTION")
print("=" * 80)

for domain in sorted(
    domain_indices
):

    total_domain = len(
        domain_indices[domain]
    )

    n_unanswerable = len(
        unanswerable_by_domain[
            domain
        ]
    )

    n_single = len(
        single_by_domain[
            domain
        ]
    )

    n_multi = len(
        multi_by_domain[
            domain
        ]
    )

    print(
        f"{domain:15s} | "
        f"total={total_domain:5,d} | "
        f"unans={n_unanswerable:5,d} | "
        f"single={n_single:5,d} | "
        f"multi={n_multi:5,d}"
    )


# ============================================================
# 14. ALLOCATE MIXED ACROSS DOMAINS
# ============================================================
#
# Mixed:
# - tổng target = 2200
# - chia gần theo tỷ lệ kích thước domain
# - không vượt quá số multi-claim của domain
# ============================================================

multi_capacity_by_domain = {
    domain: len(
        multi_by_domain[domain]
    )
    for domain in domain_indices
}

mixed_quota_by_domain = (
    allocate_proportionally(
        total_target=min(
            TARGET_MIXED,
            len(multi_claim_indices)
        ),
        group_sizes=domain_sizes,
        capacities=multi_capacity_by_domain
    )
)


# ============================================================
# 15. SELECT MIXED WITHIN EACH DOMAIN
# ============================================================
#
# Ưu tiên:
# 4 claims > 3 claims > 2 claims
#
# Nếu cùng claim count:
# random do shuffle trước sort.
# ============================================================

mixed_indices = []

remaining_multi_by_domain = defaultdict(
    list
)

for domain in sorted(
    domain_indices
):

    candidates = list(
        multi_by_domain[domain]
    )

    # Random tie breaker
    random.shuffle(candidates)

    # Claim càng nhiều càng ưu tiên mixed
    candidates.sort(
        key=lambda idx: (
            claim_count_map[idx]
        ),
        reverse=True
    )

    quota = mixed_quota_by_domain.get(
        domain,
        0
    )

    selected = candidates[:quota]

    remaining = candidates[quota:]

    mixed_indices.extend(
        selected
    )

    remaining_multi_by_domain[
        domain
    ] = remaining


print()
print("=" * 80)
print("MIXED ALLOCATION BY DOMAIN")
print("=" * 80)

for domain in sorted(
    domain_indices
):

    quota = mixed_quota_by_domain.get(
        domain,
        0
    )

    domain_total = domain_sizes[
        domain
    ]

    ratio = (
        quota
        / domain_total
        * 100
        if domain_total
        else 0
    )

    print(
        f"{domain:15s}: "
        f"{quota:5,d} mixed "
        f"({ratio:6.2f}% domain)"
    )


print("-" * 80)

print(
    f"TOTAL MIXED: "
    f"{len(mixed_indices):,} "
    f"({len(mixed_indices) / len(data) * 100:.2f}%)"
)


# ============================================================
# 16. SHOW CLAIM DISTRIBUTION OF MIXED
# ============================================================

mixed_claim_distribution = Counter(
    claim_count_map[idx]
    for idx in mixed_indices
)

print()
print("MIXED CLAIM COUNT DISTRIBUTION")
print("-" * 80)

for claim_count, count in sorted(
    mixed_claim_distribution.items(),
    reverse=True
):

    print(
        f"{claim_count:2d} claims : "
        f"{count:5,d}"
    )


# ============================================================
# 17. PRIMARY LABEL STORAGE
# ============================================================

primary_labels = [
    None
] * len(data)


# ============================================================
# 18. FORCE UNANSWERABLE -> EXTRINSIC
# ============================================================

for idx in unanswerable_indices:

    primary_labels[idx] = (
        LABEL_EXTRINSIC
    )


# ============================================================
# 19. ASSIGN MIXED
# ============================================================

for idx in mixed_indices:

    primary_labels[idx] = (
        LABEL_MIXED
    )


# ============================================================
# 20. BALANCE INTRINSIC / EXTRINSIC PER DOMAIN
# ============================================================
#
# Với từng domain:
#
#   non_mixed = total_domain - mixed
#
# target:
#   intrinsic ≈ extrinsic ≈ non_mixed / 2
#
# Nhưng unanswerable đã bị ép extrinsic.
#
# Vì vậy:
# - giữ forced extrinsic
# - answerable còn lại dùng để bù intrinsic trước
# - rồi extrinsic
#
# ============================================================

for domain in sorted(
    domain_indices
):

    # --------------------------------------------------------
    # Domain totals
    # --------------------------------------------------------

    total_domain = len(
        domain_indices[domain]
    )

    mixed_domain = (
        mixed_quota_by_domain.get(
            domain,
            0
        )
    )

    forced_extrinsic = len(
        unanswerable_by_domain[
            domain
        ]
    )

    non_mixed_domain = (
        total_domain
        - mixed_domain
    )

    # --------------------------------------------------------
    # Desired 50 / 50 intrinsic-extrinsic
    # --------------------------------------------------------

    target_intrinsic = (
        non_mixed_domain // 2
    )

    target_extrinsic = (
        non_mixed_domain
        - target_intrinsic
    )

    # --------------------------------------------------------
    # Remaining answerable samples
    # --------------------------------------------------------

    remaining_candidates = (
        list(
            single_by_domain[
                domain
            ]
        )
        +
        list(
            remaining_multi_by_domain[
                domain
            ]
        )
    )

    random.shuffle(
        remaining_candidates
    )

    # --------------------------------------------------------
    # Extrinsic already forced by unanswerable
    # --------------------------------------------------------

    need_extrinsic = max(
        0,
        target_extrinsic
        - forced_extrinsic
    )

    # --------------------------------------------------------
    # Intrinsic target
    # --------------------------------------------------------

    need_intrinsic = (
        target_intrinsic
    )

    # --------------------------------------------------------
    # Feasibility correction
    # --------------------------------------------------------

    available = len(
        remaining_candidates
    )

    # Nếu forced extrinsic quá lớn,
    # không thể đạt target 50/50.
    # Khi đó phần còn lại ưu tiên intrinsic.
    if (
        need_intrinsic
        + need_extrinsic
        > available
    ):

        need_extrinsic = max(
            0,
            available
            - need_intrinsic
        )

    # --------------------------------------------------------
    # Assign intrinsic
    # --------------------------------------------------------

    cursor = 0

    intrinsic_indices = (
        remaining_candidates[
            cursor:
            cursor + need_intrinsic
        ]
    )

    for idx in intrinsic_indices:

        primary_labels[idx] = (
            LABEL_INTRINSIC
        )

    cursor += len(
        intrinsic_indices
    )

    # --------------------------------------------------------
    # Assign extrinsic
    # --------------------------------------------------------

    extrinsic_indices = (
        remaining_candidates[
            cursor:
            cursor + need_extrinsic
        ]
    )

    for idx in extrinsic_indices:

        primary_labels[idx] = (
            LABEL_EXTRINSIC
        )

    cursor += len(
        extrinsic_indices
    )

    # --------------------------------------------------------
    # Handle leftovers
    #
    # Luôn assign vào label đang ít hơn
    # TRONG DOMAIN đó.
    # --------------------------------------------------------

    leftovers = (
        remaining_candidates[
            cursor:
        ]
    )

    for idx in leftovers:

        current_domain_counts = Counter(
            primary_labels[j]
            for j in domain_indices[
                domain
            ]
            if (
                primary_labels[j]
                is not None
            )
        )

        n_intrinsic = (
            current_domain_counts[
                LABEL_INTRINSIC
            ]
        )

        n_extrinsic = (
            current_domain_counts[
                LABEL_EXTRINSIC
            ]
        )

        if n_intrinsic <= n_extrinsic:

            primary_labels[idx] = (
                LABEL_INTRINSIC
            )

        else:

            primary_labels[idx] = (
                LABEL_EXTRINSIC
            )


# ============================================================
# 21. VALIDATE ALL ASSIGNED
# ============================================================

missing_indices = [
    idx
    for idx, label
    in enumerate(primary_labels)
    if label is None
]

if missing_indices:

    raise ValueError(
        f"Còn {len(missing_indices)} "
        f"sample chưa được gán primary label."
    )


# ============================================================
# 22. VALIDATE UNANSWERABLE
# ============================================================

bad_unanswerable = [
    idx
    for idx in unanswerable_indices
    if (
        primary_labels[idx]
        != LABEL_EXTRINSIC
    )
]

if bad_unanswerable:

    raise ValueError(
        f"Có {len(bad_unanswerable)} "
        f"unanswerable sample không phải extrinsic."
    )


# ============================================================
# 23. VALIDATE MIXED
# ============================================================

bad_mixed = [
    idx
    for idx in mixed_indices
    if claim_count_map[idx] < 2
]

if bad_mixed:

    raise ValueError(
        f"Có {len(bad_mixed)} "
        f"mixed sample có < 2 claims."
    )


# ============================================================
# 24. CREATE CLEAN OUTPUT
# ============================================================

output = []

for idx, item in enumerate(data):

    new_item = {
        key: item.get(key)
        for key in KEEP_FIELDS
    }

    primary = primary_labels[
        idx
    ]

    new_item[
        "target_hallucination_label_priority"
    ] = build_priority(
        primary
    )

    output.append(
        new_item
    )


# ============================================================
# 25. GLOBAL DISTRIBUTION
# ============================================================

primary_counts = Counter(
    primary_labels
)

total = len(data)

print()
print("=" * 80)
print("FINAL GLOBAL PRIMARY LABEL DISTRIBUTION")
print("=" * 80)

for label in ALL_LABELS:

    count = primary_counts[
        label
    ]

    pct = (
        count
        / total
        * 100
    )

    print(
        f"{label:28s}: "
        f"{count:6,d} "
        f"({pct:6.2f}%)"
    )

print("-" * 80)

print(
    f"{'TOTAL':28s}: "
    f"{total:6,d} "
    f"(100.00%)"
)


# ============================================================
# 26. DOMAIN LABEL DISTRIBUTION
# ============================================================

print()
print("=" * 110)
print("PRIMARY LABEL DISTRIBUTION BY DOMAIN")
print("=" * 110)

header = (
    f"{'Domain':15s}"
    f"{'Total':>8s}"
    f"{'Intrinsic':>18s}"
    f"{'Extrinsic':>18s}"
    f"{'Mixed':>18s}"
    f"{'I-E diff':>12s}"
)

print(header)
print("-" * 110)

domain_stats = {}

for domain in sorted(
    domain_indices
):

    indices = domain_indices[
        domain
    ]

    counts = Counter(
        primary_labels[idx]
        for idx in indices
    )

    n_total = len(indices)

    n_i = counts[
        LABEL_INTRINSIC
    ]

    n_e = counts[
        LABEL_EXTRINSIC
    ]

    n_m = counts[
        LABEL_MIXED
    ]

    domain_stats[
        domain
    ] = {
        "total": n_total,
        "intrinsic": n_i,
        "extrinsic": n_e,
        "mixed": n_m,
    }

    print(
        f"{domain:15s}"
        f"{n_total:8,d}"
        f"{n_i:8,d} ({n_i/n_total*100:5.1f}%)"
        f"{n_e:8,d} ({n_e/n_total*100:5.1f}%)"
        f"{n_m:8,d} ({n_m/n_total*100:5.1f}%)"
        f"{abs(n_i-n_e):12,d}"
    )


# ============================================================
# 27. UNANSWERABLE DISTRIBUTION BY DOMAIN
# ============================================================

print()
print("=" * 80)
print("UNANSWERABLE BY DOMAIN")
print("=" * 80)

for domain in sorted(
    domain_indices
):

    n = len(
        unanswerable_by_domain[
            domain
        ]
    )

    total_domain = len(
        domain_indices[
            domain
        ]
    )

    pct = (
        n / total_domain * 100
        if total_domain
        else 0
    )

    print(
        f"{domain:15s}: "
        f"{n:5,d} "
        f"({pct:6.2f}%)"
    )


# ============================================================
# 28. MIXED CLAIM COUNT BY DOMAIN
# ============================================================

print()
print("=" * 80)
print("MIXED CLAIM INFORMATION BY DOMAIN")
print("=" * 80)

for domain in sorted(
    domain_indices
):

    mixed_domain_indices = [
        idx
        for idx in domain_indices[
            domain
        ]
        if (
            primary_labels[idx]
            == LABEL_MIXED
        )
    ]

    if not mixed_domain_indices:
        continue

    avg_claims = sum(
        claim_count_map[idx]
        for idx
        in mixed_domain_indices
    ) / len(
        mixed_domain_indices
    )

    max_claims = max(
        claim_count_map[idx]
        for idx
        in mixed_domain_indices
    )

    min_claims = min(
        claim_count_map[idx]
        for idx
        in mixed_domain_indices
    )

    print(
        f"{domain:15s}: "
        f"mixed={len(mixed_domain_indices):5,d} | "
        f"avg_claims={avg_claims:.2f} | "
        f"min={min_claims} | "
        f"max={max_claims}"
    )


# ============================================================
# 29. FINAL VALIDATION
# ============================================================

print()
print("=" * 80)
print("VALIDATION")
print("=" * 80)

print(
    "Total samples:",
    len(data)
)

print(
    "Assigned samples:",
    len(primary_labels)
)

print(
    "Bad unanswerable:",
    len(bad_unanswerable)
)

print(
    "Bad mixed (<2 claims):",
    len(bad_mixed)
)

print(
    "Mixed assigned:",
    primary_counts[
        LABEL_MIXED
    ]
)

print(
    "Global intrinsic-extrinsic difference:",
    abs(
        primary_counts[
            LABEL_INTRINSIC
        ]
        -
        primary_counts[
            LABEL_EXTRINSIC
        ]
    )
)


# ============================================================
# 30. SHOW EXAMPLES
# ============================================================

print()
print("=" * 80)
print("SOME MIXED EXAMPLES")
print("=" * 80)

shown = 0

for idx in mixed_indices:

    item = data[idx]

    print()

    print(
        "Domain:",
        item.get("domain")
    )

    print(
        "Claim count:",
        claim_count_map[idx]
    )

    print(
        "Question:",
        item.get("question")
    )

    print(
        "Answer:",
        item.get("answer")
    )

    print(
        "Priority:",
        output[idx][
            "target_hallucination_label_priority"
        ]
    )

    shown += 1

    if shown >= 10:
        break


# ============================================================
# 31. SAVE OUTPUT
# ============================================================

OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


print()
print("=" * 80)
print("DONE")
print("=" * 80)

print(
    f"Saved -> "
    f"{OUTPUT_FILE.resolve()}"
)

Loaded: 14,883 samples

DOMAIN DISTRIBUTION
--------------------------------------------------------------------------------
education           :  2,244 ( 15.08%)
finance             :  2,567 ( 17.25%)
legal               :  2,136 ( 14.35%)
medical             :  2,339 ( 15.72%)
news                :  2,853 ( 19.17%)
science             :  2,744 ( 18.44%)

SAMPLE TYPE DISTRIBUTION
Unanswerable : 1,927
1 claim      : 7,103
>= 2 claims  : 5,853

DOMAIN / CLAIM DISTRIBUTION
education       | total=2,244 | unans=  291 | single=1,068 | multi=  885
finance         | total=2,567 | unans=  340 | single=1,206 | multi=1,021
legal           | total=2,136 | unans=  225 | single=  964 | multi=  947
medical         | total=2,339 | unans=  295 | single=1,041 | multi=1,003
news            | total=2,853 | unans=  387 | single=1,508 | multi=  958
science         | total=2,744 | unans=  389 | single=1,316 | multi=1,039

MIXED ALLOCATION BY DOMAIN
education      :   377 mixed ( 16.80% domain)
finance    

In [23]:
from collections import Counter

primary_counts = Counter(
    item["target_hallucination_label_priority"][0]
    for item in output
)

total = len(output)

print("=" * 60)
print("PRIMARY LABEL DISTRIBUTION")
print("=" * 60)

for label in [
    "intrinsic_hallucination",
    "extrinsic_hallucination",
    "mixed_hallucination",
]:
    count = primary_counts.get(label, 0)
    pct = count / total * 100 if total else 0

    print(f"{label:28s}: {count:5d} ({pct:6.2f}%)")

print("-" * 60)
print(f"{'TOTAL':28s}: {total:5d} (100.00%)")

PRIMARY LABEL DISTRIBUTION
intrinsic_hallucination     :  6190 ( 41.59%)
extrinsic_hallucination     :  6193 ( 41.61%)
mixed_hallucination         :  2500 ( 16.80%)
------------------------------------------------------------
TOTAL                       : 14883 (100.00%)


In [26]:
import pandas as pd 

df = pd.read_json("data/guided_assigned_clean.json")
len(df)

14883

# bỏ bớt

In [1]:
import json
from pathlib import Path


INPUT_FILE = Path("data/guided_assigned_label.json")
QWEN_FILE = Path("data/output_qwen/qwen3_8_output.json")
OUTPUT_FILE = Path("data/guided_assigned_deepseek.json")


# ============================================================
# LOAD
# ============================================================

with INPUT_FILE.open("r", encoding="utf-8") as f:
    data = json.load(f)

with QWEN_FILE.open("r", encoding="utf-8") as f:
    qwen_data = json.load(f)


print(f"Original samples : {len(data):,}")
print(f"Qwen output      : {len(qwen_data):,}")


# ============================================================
# QWEN ĐÃ CÓ HALLUCINATED RESPONSE
# ============================================================

qwen_completed_ids = {
    item["chunk_id"]
    for item in qwen_data
    if (
        item.get("chunk_id")
        and item.get("qwen_generation", {})
                .get("result", {})
                .get("hallucinated_response")
    )
}


print(f"Qwen có hallu answer : {len(qwen_completed_ids):,}")


# ============================================================
# GIỮ LẠI NHỮNG MẪU QWEN CHƯA SINH HALLU ANSWER
# ============================================================

deepseek_data = [
    item
    for item in data
    if item.get("chunk_id") not in qwen_completed_ids
]


# ============================================================
# SAVE
# ============================================================

with OUTPUT_FILE.open("w", encoding="utf-8") as f:
    json.dump(
        deepseek_data,
        f,
        ensure_ascii=False,
        indent=2,
    )


print("=" * 70)
print(f"Qwen đã làm       : {len(qwen_completed_ids):,}")
print(f"DeepSeek cần làm  : {len(deepseek_data):,}")
print(f"Tổng              : {len(data):,}")
print(f"Saved             : {OUTPUT_FILE}")
print("=" * 70)

Original samples : 14,883
Qwen output      : 14,883
Qwen có hallu answer : 533
Qwen đã làm       : 533
DeepSeek cần làm  : 13,970
Tổng              : 14,883
Saved             : data/guided_assigned_deepseek.json
